In [ ]:
import tkinter as tk
from tkinter import messagebox
import numpy as np
import pandas as pd
import pickle
import bcrypt
import os
import re
import webbrowser
from sklearn.preprocessing import MinMaxScaler

USER_DATA_FILE = "users.csv"

# Load the trained model
with open("autism_model.pkl", "rb") as file:
    model = pickle.load(file)

original_features = ['age', 'gender', 'ethnicity', 'jundice', 'austim', 'contry_of_res', 'result', 'relation',
                     'A1_Score', 'A2_Score', 'A3_Score', 'A4_Score', 'A5_Score', 'A6_Score', 'A7_Score', 'A8_Score',
                     'A9_Score', 'A10_Score', 'm_m']

# Password requirements (at least 8 characters, 1 uppercase, 1 lowercase, 1 number, and 1 special character)
PASSWORD_REGEX = r"^(?=.*[a-z])(?=.*[A-Z])(?=.*\d)(?=.*[@$!%*?&])[A-Za-z\d@$!%*?&]{8,}$"

descriptions = {
    "age":"Age of the individual.",
    "gender":"m(male) or f(female).",
    "ethnicity":"racial background of the individual.",
    "jundice":"Presence or absense of jaundice in the individual(yes/no)?",
    "contry_of_res":"Country of residence of the individual.",
    "result":"add 1 if you are suffering with - Frequent Fever,Digestive Problems,Sleep Disturbances,Skin Sensitivities,Increased Allergies,Chronic Fatigue,etc",
    "relation":"Self/parent/caregiver.",
    "austim":"Is anyone in u r family diagnosis of autism(yes/no)?",
    "A1_Score": "Are u Heightened Sensory Sensitivity?",
    "A2_Score": "Are u Big-Picture Thinker?",
    "A3_Score": "Do u have Multitasking Ability?",
    "A4_Score": "Are u Flexibile in Activities?",
    "A5_Score": "Do u have Difficulty in Peer Conversations?",
    "A6_Score": "Do u have Problem in Small Talks?",
    "A7_Score": "Do u have Theory of Mind Challenges?",
    "A8_Score": "Do u have Imaginative Play in Preschool.",
    "A9_Score": "Do u haveEmpathy through Facial Expressions.",
    "A10_Score": "Do u have Difficulty in Social Initiation.",
    "m_m":"Is u r parents maternal marriage(0-no/1-yes).",
}

def hash_password(password):
    return bcrypt.hashpw(password.encode(), bcrypt.gensalt()).decode()

def verify_password(password, hashed):
    return bcrypt.checkpw(password.encode(), hashed.encode())

def is_valid_password(password):
    return bool(re.match(PASSWORD_REGEX, password))

# Ensure user data file exists
if not os.path.exists(USER_DATA_FILE):
    with open(USER_DATA_FILE, "w") as f:
        f.write("username,password\n")

def open_guidance_page():
    webbrowser.open("guidance.html")

def register_user():
    username = username_entry.get()
    password = password_entry.get()
    
    if not username or not password:
        messagebox.showerror("Error", "Username and password cannot be empty!")
        return
    
    if not is_valid_password(password):
        messagebox.showerror("Error", "Password must be at least 8 characters long, contain an uppercase letter, a lowercase letter, a number, and a special character.")
        return

    with open(USER_DATA_FILE, "r") as f:
        users = [line.strip().split(",") for line in f.readlines()]
    
    if any(user[0] == username for user in users):
        messagebox.showerror("Error", "Username already exists!")
        return
    
    hashed_password = hash_password(password)
    with open(USER_DATA_FILE, "a") as f:
        f.write(f"{username},{hashed_password}\n")
    
    messagebox.showinfo("Success", "Registration successful! Please log in.")
    register_window.destroy()
    show_login_screen()
def login_user():
    global username_entry, password_entry  # Ensure they are properly referenced
    try:
        username = username_entry.get()
        password = password_entry.get()
        
        if not username or not password:
            messagebox.showerror("Error", "Username and password cannot be empty!")
            return

        with open(USER_DATA_FILE, "r") as f:
            users = [line.strip().split(",") for line in f.readlines()]
        
        for user in users:
            if user[0] == username and verify_password(password, user[1]):
                messagebox.showinfo("Success", "Login successful!")
                login_window.destroy()
                show_main_screen()
                return
        
        messagebox.showerror("Error", "Invalid username or password!\n or\naccount does not exist, do registe!")
    
    except Exception as e:
        messagebox.showerror("Error", f"Unexpected error: {str(e)}")


def show_login_screen():
    global login_window, username_entry, password_entry  # Ensure they are global
    login_window = tk.Tk()
    login_window.title("Login")
    login_window.geometry("400x300")

    tk.Label(login_window, text="Username:").pack()
    username_entry = tk.Entry(login_window)
    username_entry.pack()

    tk.Label(login_window, text="Password:").pack()
    password_entry = tk.Entry(login_window, show="*")
    password_entry.pack()

    tk.Button(login_window, text="Login", command=login_user).pack()
    tk.Button(login_window, text="Register", command=show_register_screen).pack()

    login_window.mainloop()

def show_register_screen():
    global register_window, username_entry, password_entry
    register_window = tk.Tk()
    register_window.title("Register")
    register_window.geometry("400x300")

    tk.Label(register_window, text="Username:").pack()
    username_entry = tk.Entry(register_window)
    username_entry.pack()

    tk.Label(register_window, text="Password:").pack()
    password_entry = tk.Entry(register_window, show="*")
    password_entry.pack()

    tk.Button(register_window, text="Register", command=register_user).pack()

    register_window.mainloop()

def show_main_screen():
    root = tk.Tk()
    root.title("Autism Predictor")
    root.geometry("600x700")

    entries = {}
    for idx, feature in enumerate(original_features):
        label_text = f"{feature}: ({descriptions[feature]})" if feature in descriptions else f"{feature}:"
        label = tk.Label(root, text=label_text, wraplength=500, justify="left")
        label.grid(row=idx, column=0, padx=10, pady=5, sticky="w")
        entry = tk.Entry(root)
        entry.grid(row=idx, column=1, padx=10, pady=5)
        entries[feature] = entry

    def predict_autism():
        try:
            input_data = {feature: entries[feature].get() for feature in original_features}
            categorical_features = ['gender', 'ethnicity', 'jundice', 'austim', 'contry_of_res', 'relation']
            for feature in categorical_features:
                input_data[feature] = str(input_data[feature])
            
            numerical_features = ['age', 'result', 'm_m']
            for feature in numerical_features:
                input_data[feature] = float(input_data[feature])
            
            for i in range(1, 11):
                input_data[f"A{i}_Score"] = int(input_data[f"A{i}_Score"])
            
            input_df = pd.DataFrame([input_data])
            dataset = pd.read_csv("a.csv")
            dataset_encoded = pd.get_dummies(dataset[original_features])
            final_columns = dataset_encoded.columns
            
            input_encoded = pd.get_dummies(input_df)
            input_final = pd.DataFrame(columns=final_columns)
            input_final = pd.concat([input_final, input_encoded], ignore_index=True).fillna(0)
            
            scaler = MinMaxScaler()
            input_final[numerical_features] = scaler.fit_transform(input_final[numerical_features])
            
            input_array = input_final.to_numpy()
            prediction = model.predict(input_array)[0]

            if prediction == 1:
                messagebox.showinfo("Prediction Result", "Autism Detected. Click OK for guidance.")
                open_guidance_page()
                root.destroy()
            else:
                messagebox.showinfo("Prediction Result", "No Autism Detected. Continue regular health checkups.")
                root.destroy()
        except Exception as e:
            messagebox.showerror("Error", "Error check the inputs again!")

    tk.Button(root, text="Predict", command=predict_autism, bg="blue", fg="white").grid(row=len(original_features), column=0, columnspan=2, pady=20)
    root.mainloop()

show_login_screen()
